# Galaxy Spectra

Unlike component spectra, galaxy level spectra are by definition integrated. These spectra can only be combinations of component spectra. To see how to define emission models for galaxy level spectra see the [combined emission](../../emission_models/combined_models.ipynb) docs. Here we will demonstrate the usage of a combined model to generate spectra for a galaxy. First we define that combined model.

In [ ]:
import numpy as np
from unyt import K, Mpc, Msun, Myr, cm, yr

from synthesizer.emission_models import (
    BimodalPacmanEmission,
    Blackbody,
    DustEmission,
    EmissionModel,
    Greybody,
    UnifiedAGN,
)
from synthesizer.emission_models.attenuation import PowerLaw
from synthesizer.grid import Grid
from synthesizer.parametric import SFH, ZDist
from synthesizer.parametric import Stars as ParametricStars
from synthesizer.particle import BlackHoles, Galaxy
from synthesizer.particle.stars import sample_sfzh

# Get the grids which we'll need for extraction
grid_name = "test_grid"
grid = Grid(grid_name)
nlr_grid = Grid("test_grid_agn-nlr")
blr_grid = Grid("test_grid_agn-blr")

# Get the stellar pacman model
pc_model = BimodalPacmanEmission(
    grid=grid,
    tau_v_ism=1.0,
    tau_v_birth=0.7,
    dust_curve_ism=PowerLaw(slope=-1.3),
    dust_curve_birth=PowerLaw(slope=-0.7),
    dust_emission_ism=Blackbody(temperature=100 * K),
    dust_emission_birth=Blackbody(temperature=30 * K),
    fesc=0.2,
    fesc_ly_alpha=0.9,
    label="stellar_total",
)
pc_model.set_per_particle(True)

# Initialise the UnifiedAGN model
uniagn = UnifiedAGN(
    nlr_grid,
    blr_grid,
    ionisation_parameter_nlr=0.01,
    hydrogen_density_nlr=1e4 * cm**-3,
    ionisation_parameter_blr=0.1,
    hydrogen_density_blr=1e10 * cm**-3,
    torus_emission_model=Greybody(1000 * K, 1.5),
    diffuse_dust_curve=PowerLaw(slope=-1.0),
    per_particle=True,
)
uniagn.add_label_prefix("agn")
pc_model.add_label_prefix("stellar")


gal_intrinsic = EmissionModel(
    label="total_intrinsic",
    combine=(uniagn["agn_intrinsic"], pc_model["stellar_intrinsic"]),
    emitter="galaxy",
)

gal_attenuated = EmissionModel(
    label="total_attenuated",
    combine=(uniagn["agn_attenuated"], pc_model["stellar_attenuated"]),
    related_models=(gal_intrinsic,),
    emitter="galaxy",
)

# And now include the dust emission
gal_dust = DustEmission(
    dust_emission_model=Greybody(30 * K, 1.2),
    dust_lum_intrinsic=gal_intrinsic,
    dust_lum_attenuated=gal_attenuated,
    emitter="galaxy",
    label="dust_emission",
)

gal_total = EmissionModel(
    label="total",
    combine=(gal_attenuated, gal_dust),
    related_models=(gal_intrinsic,),
    emitter="galaxy",
)

gal_total.plot_emission_graph(show=True)

This emission model is going to produce a lot of spectra. In reality we probably don't want to carry around all these spectra on a galaxy or its components. Instead we can select exactly which spectra we want to save. This can either be done by setting the save flag to `False` on selected models like below.

In [ ]:
# Discard some spectra
gal_total["stellar_young_nebular_continuum"].set_save(False)
gal_total["stellar_old_nebular_continuum"].set_save(False)
gal_total["agn_disc_transmitted_nlr"].set_save(False)
gal_total["agn_disc_transmitted_blr"].set_save(False)
gal_total["stellar_young_attenuated_nebular"].set_save(False)
gal_total.plot_emission_graph()

But if we only want a handful of spectra saved this is pretty cumbersome. We can instead state exactly which spectra we want saved using the `save_spectra` method which takes an arbitrary number of emission model labels as arguments.

In [ ]:
gal_total.save_spectra(
    "total",
    "dust_emission",
    "total_attenuated",
    "total_intrinsic",
    "stellar_intrinsic",
    "stellar_attenuated",
)
gal_total.plot_emission_graph()

First let's make a galaxy containing both stars and black holes. We'll use a particle galaxy with particle ``Stars`` and ``BlackHoles``, but getting spectra is the same for both particle and parametric galaxies.

In [ ]:
# Define the metallicity history
zh = ZDist.DeltaConstant(metallicity=0.01)

# Define the star formation history
sfh_p = {"max_age": 100 * Myr}
sfh = SFH.Constant(**sfh_p)

# Initialise the parametric Stars object
param_stars = ParametricStars(
    grid.log10ages,
    grid.metallicities,
    sf_hist=sfh,
    metal_dist=zh,
    initial_mass=10**10 * Msun,
)

# Define the number of stellar particles we want
n = 100

# Sample the parametric SFZH, producing a particle Stars object
# we will also pass some keyword arguments for some example attributes
part_stars = sample_sfzh(
    sfzh=param_stars.sfzh,
    log10ages=param_stars.log10ages,
    log10metallicities=param_stars.log10metallicities,
    nstar=n,
    redshift=1,
)
print(part_stars)

# Make fake properties
n = 4
masses = 10 ** np.random.uniform(low=6, high=8, size=n) * Msun
coordinates = np.random.normal(0, 1.5, (n, 3)) * Mpc
accretion_rates = 10 ** np.random.uniform(low=-2, high=1, size=n) * Msun / yr
metallicities = np.full(n, 0.01)
tau_v = 0.5 * np.ones(n)

# And get the black holes object
blackholes = BlackHoles(
    masses=masses,
    coordinates=coordinates,
    accretion_rates=accretion_rates,
    metallicities=metallicities,
    tau_v=tau_v,
    ionisation_parameter_nlr=0.01,
    hydrogen_density_nlr=1e4 * cm**-3,
    ionisation_parameter_blr=0.1,
    hydrogen_density_blr=1e10 * cm**-3,
)

# And create the galaxy
galaxy = Galaxy(
    stars=part_stars,
    black_holes=blackholes,
    redshift=1,
)

Similarly to component spectra, to generate galaxy level spectra we call the ``get_spectra`` method on a ``Galaxy``, with the ``EmissionModel``.
There is no need to call the component methods; the ``Galaxy`` level method will handle generating the component spectra automatically. 

In [ ]:
spectra = galaxy.get_spectra(gal_total, out_dtype=np.float64)

Now we can plot the spectra we've generated using the ``plot_spectra`` method. Unlike the component versions of this method, we need to also flag which type of spectra we'd like to plot from the galaxy. 

We can plot only stellar spectra, only black hole spectra or the combined spectra. Below we show the combined.

In [ ]:
galaxy.plot_spectra(
    combined_spectra=True,
    stellar_spectra=True,
    black_hole_spectra=False,
    show=True,
    figsize=(6, 4),
)

## Getting observed spectra

If we want to get the observed spectra for all spectra on a galaxy we can use the ``get_observed_spectra`` method.
This uses the ``redshift`` attribute on the galaxy object by default.

In [ ]:
from astropy.cosmology import Planck18 as cosmo

galaxy.get_observed_spectra(cosmo)

Which we can also plot with the ``plot_observed_spectra`` method (here only showing the combined spectra, which is the default).

In [ ]:
galaxy.plot_observed_spectra(show=True, figsize=(8, 6))

### Including peculiar velocities (redshift-space distortions)

A galaxy's observed redshift is set by both the Hubble flow and its line-of-sight peculiar velocity, the source of redshift-space distortions (RSD) in e.g. lightcones. This can be included by passing a ``peculiar_velocity`` to ``get_observed_spectra`` (or directly to ``Sed.get_fnu``), in which case the galaxy's ``redshift`` is treated as the cosmological redshift $z$ and the spectra are shifted to the observed redshift

$$1 + z_{\rm obs} = (1 + z)(1 + v_{\rm los} / c),$$

where a positive velocity is receding (moving away from the observer). The wavelength shift and the $(1 + z)$ flux factor use $z_{\rm obs}$, while the distance to the source and the IGM attenuation are set by the cosmological $z$, since a peculiar velocity is a motion rather than a displacement (the effective luminosity distance is $d_L(z)(1 + z_{\rm obs})/(1 + z)$). The rest-frame spectra are untouched, so the shift is applied at the observed-frame step without regenerating any spectra.

The velocity must be passed with units. If ``peculiar_velocity`` is not passed, a ``peculiar_velocity`` attribute on the galaxy is used if one has been set.

In [ ]:
import matplotlib.pyplot as plt
from unyt import angstrom, km, s

# Keep the observed spectrum without a peculiar velocity for comparison
obslam_cosmo = galaxy.spectra["total"].obslam.copy()
fnu_cosmo = galaxy.spectra["total"].fnu.copy()

# Recompute the observed spectra for a galaxy receding at 1000 km/s
galaxy.get_observed_spectra(cosmo, peculiar_velocity=1000 * km / s)
sed = galaxy.spectra["total"]

# Compare the two around the redshifted H-beta and [OIII] lines
fig, ax = plt.subplots(figsize=(6, 4))
for lam, fnu, label in (
    (obslam_cosmo, fnu_cosmo, "$v_{\\rm los} = 0$"),
    (sed.obslam, sed.fnu, "$v_{\\rm los} = 1000$ km/s"),
):
    window = (lam > 9600 * angstrom) & (lam < 10200 * angstrom)
    ax.plot(lam[window], fnu[window], label=label)
ax.set_xlabel(r"$\lambda_{\rm obs} / \AA$")
ax.set_ylabel(r"$F_\nu / \rm nJy$")
ax.legend()
plt.show()

Note that this is distinct from setting ``vel_shift=True`` on an ``EmissionModel``. That Doppler shifts each particle's rest-frame spectrum by its own line-of-sight velocity during ``get_spectra``, capturing internal kinematics (e.g. rotation and dispersion) at the cost of a more expensive spectra calculation. ``peculiar_velocity`` instead applies a single bulk shift to every spectrum on the galaxy and is essentially free. The two are alternatives: spectra generated with ``vel_shift`` (and any spectra built from them) are flagged, and passing a ``peculiar_velocity`` for them raises an error rather than applying the bulk motion twice. To capture both internal kinematics and the bulk motion, use ``vel_shift`` with particle velocities that include the galaxy's bulk velocity.

## Modifying `EmissionModel` parameters with `get_spectra`

As well as modifying a model explicitly, it's also possible to override the properties of a model at the point `get_spectra` is called. These modifications will not be remembered by the model afterwards. As it stands, this form of modification is supported for the `dust_curve`, `tau_v`, `fesc`, `covering_fraction` and `masks`.

For more details on this see the [stars](stars.ipynb) and [blackhole](blackholes.ipynb) spectra docs where the modification is described and demonstrated in each context.

## Plotting spectra stacks 

If you want to visualise a 2D spectra, e.g. a per particle spectra like the stellar spectra we produced above then you can use the ``plot_spectra_stack`` method. This will produce a 2D plot of the spectra, with the wavelength on the x-axis and (by default) 100 spectra stacked on the y axis. The colour of each pixel in the plot corresponds to the flux at that wavelength for that particle.

There are 4 possible "orderings" for the y-axis of the plot:

- ``"peak"`` - From bottom to top the spectra will be ordered by their peak flux.
- ``"bolometric_luminosity"`` - From bottom to top the spectra will be ordered by their bolometric luminosity.
- ``lam`` - From bottom to top the spectra will be ordered by their luminosity at the specificed wavelength.
- ``None`` - From bottom to top the spectra will be ordered by their index in the array. This is the default behaviour.


In [ ]:
from unyt import angstrom

galaxy.stars.particle_spectra["stellar_attenuated"].plot_spectra_stack(
    order="bolometric_luminosity",
    show=True,
    xlimits=(10**3 * angstrom, 10**4.5 * angstrom),
    vmin=10**19.0,
)
galaxy.stars.particle_spectra["stellar_attenuated"].plot_spectra_stack(
    order="peak",
    show=True,
    xlimits=(10**3 * angstrom, 10**4.5 * angstrom),
    vmin=10**19.0,
)
galaxy.stars.particle_spectra["stellar_attenuated"].plot_spectra_stack(
    order=1216 * angstrom,
    show=True,
    xlimits=(10**3 * angstrom, 10**4.5 * angstrom),
    vmin=10**19.0,
)
galaxy.stars.particle_spectra["stellar_attenuated"].plot_spectra_stack(
    show=True, xlimits=(10**3 * angstrom, 10**4.5 * angstrom), vmin=10**19.0
)

## Printing Used Parameters

During spectra generation, emission models cache the parameters they extract and use from the emitter. These cached parameters can be printed in a nicely formatted table to inspect which values were actually used by each model.

In [ ]:
# Print the cached parameters used by the models
galaxy.print_used_parameters()